## Machine learning pipeline — walkthrough

**Docs:** [`docs/architecture.md`](../docs/architecture.md) (modules, data flow), [`docs/ml_guide.md`](../docs/ml_guide.md) (story + example), [`README.md`](../README.md) (setup, CLI).

### Prerequisites

- **Working directory:** start Jupyter from the **repo root** or from `notebooks/` so the first code cell can set `ROOT` and `sys.path` correctly.
- **Dependencies:** `pip install -r requirements.txt` and `python -m spacy download en_core_web_sm` (spaCy model used in Step 3).
- **LLM (optional):** put `OPENAI_API_KEY` in a `.env` file at the repo root **before Step 9** if you want live API summaries and campaign copy; without it, the step still runs but falls back where the code path expects an API.
- **First run:** Step 1 may download the Kaggle trending CSV if `data/raw/` does not already have it (needs network).

In [1]:
import os
import sys
from pathlib import Path

_here = Path.cwd().resolve()
if (_here / "src").is_dir():
    ROOT = _here
elif (_here.parent / "src").is_dir():
    ROOT = _here.parent
else:
    raise RuntimeError("Start Jupyter from the repo root or from notebooks/.")

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Repository root:", ROOT)

Repository root: /Users/s0s0rn1/Desktop/Intuit/Mailchimp-Email-Marketing


In [2]:
from dotenv import load_dotenv

from IPython.display import display

import pandas as pd

from src.config.settings import Settings
from src.constants.pipeline_io import (
    TOPIC_INSIGHTS_FILENAME,
    VIDEOS_TEXT_BEFORE_TOPICS_FILENAME,
    VIDEOS_WITH_TOPICS_FILENAME,
)
from src.pipeline.pipeline_run import (
    TrendPipelineContext,
    _step_assign_topics,
    _step_enrich_documents,
    _step_load_dataset,
    _step_prepare_documents,
    _step_save_final_artifacts,
    _step_save_text_prep_checkpoint,
    _step_attach_topic_keywords,
    _step_marketer_insights,
    _step_offline_ranking_evaluation,
    _step_trend_scoring,
)
from src.pipeline.trend_engine import TrendPipelineEngine

load_dotenv(ROOT / ".env")
pd.set_option("display.max_colwidth", 72)

### Configuration and engine

**What:** `Settings` holds paths, `max_rows`, model names, `llm_top_n`, `log_ranking_evaluation`, etc. `TrendPipelineEngine` wires loaders, NLP, BERTopic, scoring, evaluation, and the insight client—same as `main.py`.

**Why `max_rows` here:** smaller subsamples make the notebook faster for teaching; raise or remove the cap for a full pass.


In [3]:
settings = Settings(max_rows=800)
engine = TrendPipelineEngine(settings)

print("max_rows:", settings.max_rows)
print("processed_data_dir:", settings.processed_data_dir)
print("output_dir:", settings.output_dir)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


max_rows: 800
processed_data_dir: data/processed
output_dir: outputs


### Pipeline context

**What:** `TrendPipelineContext` is a small bag: `engine`, `videos_df` (one row per video, growing columns as steps run), and `topic_insights_df` (one row per topic, filled from Step 6 onward).

**Why:** each `_step_*` mutates `ctx` in place—same pattern as `run_trend_pipeline()` in code. **Run steps in order**; skipping or reordering breaks downstream assumptions.


In [4]:
ctx = TrendPipelineContext(engine=engine)

#### Step 1: Load dataset

**What:** Bring trending video rows into a pandas `DataFrame`.

**How:** `TrendingDatasetLoader` reads the regional CSV (default: Kaggle `USvideos.csv`), keeps schema columns, and `validate_trending_video_rows` checks types/required fields.

**Why:** Everything downstream assumes clean, validated tabular input; no embeddings or topics yet.

The preview lists key columns later used in scoring and enrichment: `views`, `likes`, `dislikes`, `comment_count`, `trending_date`, and `publish_time` feed **`TrendScorer.score`** in Step 6 (with `topic` from Step 5). `description` appears only if enabled in `Settings`.


In [5]:
_step_load_dataset(ctx)

v = ctx.videos_df
assert v is not None
print("videos_df shape:", v.shape)
cols = [
    c
    for c in (
        "title",
        "tags",
        "views",
        "likes",
        "dislikes",
        "comment_count",
        "trending_date",
        "publish_time",
        "description",
    )
    if c in v.columns
]
display(v[cols].head(2))


Loading dataset from: data/raw/USvideos.csv
videos_df shape: (800, 8)


,title,tags,views,likes,dislikes,comment_count,trending_date,publish_time
0,WE WANT TO TALK ABOUT OUR MARRIAGE,SHANtell martin,748374,57527,2966,15954,17.14.11,2017-11-13T17:13:01.000Z
1,The Trump Presidency: Last Week Tonight with John Oliver (HBO),"last week tonight trump presidency|""last week tonight donald trump""|...",2418783,97185,6146,12703,17.14.11,2017-11-13T07:30:00.000Z


#### Step 2: Build documents

**What:** One text field per video for NLP.

**How:** Concatenate **title**, **tags**, and optionally **description** (`Settings.use_description`) into a column `document`.

**Why:** Clustering and embeddings need a single string per row; this matches how `main.py` builds the document.


In [6]:
_step_prepare_documents(ctx)

v = ctx.videos_df
cols = [c for c in ("title", "document") if c in v.columns]
display(v[cols].head(2))

,title,document
0,WE WANT TO TALK ABOUT OUR MARRIAGE,WE WANT TO TALK ABOUT OUR MARRIAGE SHANtell martin
1,The Trump Presidency: Last Week Tonight with John Oliver (HBO),The Trump Presidency: Last Week Tonight with John Oliver (HBO) last ...


#### Step 3: spaCy normalization (lemmatized text)

**What:** Turn raw `document` into lighter, more comparable text.

**How:** `SpacyPreprocessor` lemmatizes, drops stopwords/noise, and writes `cleaned_text`.

**Why:** Reduces spelling/noise so sentence embeddings and BERTopic’s bag-of-words statistics are more stable; **embeddings use `cleaned_text`**, not the original title alone.


In [7]:
_step_enrich_documents(ctx)

v = ctx.videos_df
cols = [c for c in ("title", "document", "cleaned_text") if c in v.columns]
display(v[cols].head(2))

,title,document,cleaned_text
0,WE WANT TO TALK ABOUT OUR MARRIAGE,WE WANT TO TALK ABOUT OUR MARRIAGE SHANtell martin,want talk marriage shantell martin
1,The Trump Presidency: Last Week Tonight with John Oliver (HBO),The Trump Presidency: Last Week Tonight with John Oliver (HBO) last ...,trump presidency week tonight john oliver hbo week tonight trump pre...


#### Step 4: Save text-prep checkpoint (before topics)

**What:** Persist the video table **after** text prep, **before** expensive embedding/topic work.

**How:** Writes `videos_text_before_topics.csv` under `processed_data_dir`.

**Why:** Lets you **reuse** `document` / `cleaned_text` without re-running spaCy, and makes runs **auditable** (diff/checkpoint).


In [8]:
_step_save_text_prep_checkpoint(ctx)

ck = Path(settings.processed_data_dir) / VIDEOS_TEXT_BEFORE_TOPICS_FILENAME
print("Checkpoint:", ck.resolve(), "exists:", ck.is_file())

Checkpoint: /Users/s0s0rn1/Desktop/Intuit/Mailchimp-Email-Marketing/data/processed/videos_text_before_topics.csv exists: True


#### Step 5: Embeddings + topic assignment

**What:** Assign each video to a **topic cluster** (integer id) and a confidence.

**How:** `SentenceTransformer` encodes `cleaned_text` into vectors; **BERTopic** (`TopicModeler.fit_transform`) clusters in embedding space and fits topic representations. Rows get `topic` and `topic_confidence`. **−1** = outlier/noise bucket (excluded from topic-level scoring later).

**Why:** Groups “similar” videos for aggregate trends; **clustering follows embeddings**; keyword statistics inside BERTopic still use the text side of the pipeline.


In [9]:
_step_assign_topics(ctx)

v = ctx.videos_df
cols = [c for c in ("title", "topic", "topic_confidence") if c in v.columns]
display(v[cols].head(5))

Batches:   0%|          | 0/13 [00:00<?, ?it/s]

2026-04-20 09:55:01,801 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-20 09:55:07,710 - BERTopic - Dimensionality - Completed ✓
2026-04-20 09:55:07,710 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-20 09:55:07,741 - BERTopic - Cluster - Completed ✓
2026-04-20 09:55:07,743 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-04-20 09:55:07,768 - BERTopic - Representation - Completed ✓


,title,topic,topic_confidence
0,WE WANT TO TALK ABOUT OUR MARRIAGE,1,0.500068
1,The Trump Presidency: Last Week Tonight with John Oliver (HBO),1,0.857786
2,"Racist Superman | Rudy Mancuso, King Bach & Lele Pons",6,0.440868
3,Nickelback Lyrics: Real or Fake?,-1,0.022555
4,I Dare You: GOING BALD!?,1,0.568101


#### Step 6: Trend scoring

**What:** One row per **topic** with final LambdaMART-based ranking outputs used by the dashboard/LLM flow.

**How:** `TrendScorer.score` computes topic features, assigns each topic to a ranking segment, trains LambdaMART on **(`date`, `ranking_segment`)** query groups using segment-local labels, then returns final topic-level ranking fields (including `ranking_segment`, `segment_rank`, and `trend_score`).

**Why:** This table is the **final scored output** (for serving), while Step 6.1 shows the **training/query construction and pairwise ordering** details.


In [10]:
_step_trend_scoring(ctx)

t = ctx.topic_insights_df
assert t is not None
cols = [
    "topic",
    "ranking_segment",
    "segment_rank",
    "trend_score",
    "volume",
    "momentum",
    "avg_proxy_ctr_recency",
    "avg_views",
    "avg_likes",
]
cols = [c for c in cols if c in t.columns]
display(t[cols].head(5))

,topic,ranking_segment,segment_rank,trend_score,volume,momentum,avg_proxy_ctr_recency,avg_views,avg_likes
0,0,entertainment,1.0,1.000000,133,-0.034483,0.026410,2.295221e+06,108496.203008
1,8,general,1.0,0.976306,42,0.272727,0.029558,6.187073e+05,33974.190476
2,0,technology,1.0,0.967521,133,-0.034483,0.026410,2.295221e+06,108496.203008
3,8,food,1.0,0.952986,42,0.272727,0.029558,6.187073e+05,33974.190476
4,11,seasonal,1.0,0.915100,29,0.142857,0.033452,1.305098e+06,28731.620690


#### Step 6.1: LambdaMART flow for one multi-segment demo topic

**What:** Trace ranking flow with a topic that appears in **multiple segments** in this run.

**How:** We auto-pick a topic with the highest segment diversity, then show (1) its topic-segment rows, (2) its per-date training rows as topic-segment pairs, and (3) one `(date, segment)` query where pairwise ordering is formed.

**Why:** This makes the topic-segment pair flow explicit for demo audiences.


In [18]:
# Build the exact LambdaMART training frame used by TrendScorer (compact demo walkthrough).
scorer = ctx.engine.trend_scorer
v = ctx.videos_df
assert v is not None

valid = v[v["topic"] != -1].copy()
valid["date"] = scorer.parse_trending_date(valid["trending_date"])
valid = valid.dropna(subset=["date"])
valid = scorer._build_video_level_features(valid)
valid = scorer._attach_video_segments(valid, ctx.engine.topic_modeler)

# 1) Pick one demo topic (prefer one that appears across multiple segments).
topic_segment_counts = (
    valid.groupby("topic")["ranking_segment"].nunique().sort_values(ascending=False)
)
if topic_segment_counts.empty:
    raise RuntimeError("No valid topics available for LambdaMART walkthrough.")

demo_topic_id = int(topic_segment_counts.index[0])
n_segments = int(topic_segment_counts.iloc[0])
print("chosen demo topic:", demo_topic_id)
print("segment variants for demo topic:", n_segments)
if n_segments <= 1:
    print("Note: no multi-segment topic in this slice; using highest-coverage topic.")

# 2) Show only the core training rows for this topic as topic-segment pairs.
train_df = scorer._build_lambdamart_training_frame(valid)
demo_train_rows = train_df[train_df["topic"] == demo_topic_id][
    [
        "date",
        "ranking_segment",
        "topic",
        "doc_count",
        "label_gain",
        "label_relevance",
    ]
].sort_values(["date", "ranking_segment"])
print("demo topic training rows:", len(demo_train_rows))
display(demo_train_rows)

# 3) Build one query from this topic and show pairwise ordering table.
demo_queries = demo_train_rows[["date", "ranking_segment"]].drop_duplicates()
if demo_queries.empty:
    print("No LambdaMART query rows available for this demo topic in this slice.")
else:
    example_date, example_segment = demo_queries.sort_values("date").iloc[-1]
    print(f"Example query for demo topic: date={example_date}, segment={example_segment}")

    query_rows = train_df[
        (train_df["date"] == example_date)
        & (train_df["ranking_segment"] == example_segment)
    ][
        [
            "topic",
            "label_relevance",
            "label_gain",
        ]
    ].sort_values("label_relevance", ascending=False).reset_index(drop=True)

    pairs = query_rows.merge(query_rows, how="cross", suffixes=("_a", "_b"))
    pairs = pairs[
        (pairs["topic_a"] != pairs["topic_b"])
        & (pairs["label_relevance_a"] > pairs["label_relevance_b"])
    ]

    pair_cols = [
        "topic_a",
        "label_relevance_a",
        "label_gain_a",
        "topic_b",
        "label_relevance_b",
        "label_gain_b",
    ]
    display(pairs[pair_cols].head(12))
    print(f"Pairwise ordering comparisons in this query: {len(pairs)}")


chosen demo topic: 1
segment variants for demo topic: 4
demo topic training rows: 10


,date,ranking_segment,topic,doc_count,label_gain,label_relevance
4,2017-11-14,entertainment,1,3,0.000000,1
9,2017-11-14,general,1,17,0.550270,2
28,2017-11-15,entertainment,1,3,0.190909,1
34,2017-11-15,general,1,20,0.548904,3
41,2017-11-15,technology,1,2,0.300000,4
52,2017-11-16,entertainment,1,2,0.080138,1
57,2017-11-16,general,1,19,0.260217,2
64,2017-11-16,technology,1,2,0.112807,1
82,2017-11-17,general,1,16,0.166802,1
89,2017-11-17,technology,1,5,0.300000,0


Example query for demo topic: date=2017-11-17 00:00:00, segment=technology


,topic_a,label_relevance_a,label_gain_a,topic_b,label_relevance_b,label_gain_b
1,5,4,0.700000,2,3,0.642543
2,5,4,0.700000,7,2,0.547787
3,5,4,0.700000,6,1,0.394261
4,5,4,0.700000,1,0,0.300000
7,2,3,0.642543,7,2,0.547787
8,2,3,0.642543,6,1,0.394261
9,2,3,0.642543,1,0,0.300000
13,7,2,0.547787,6,1,0.394261
14,7,2,0.547787,1,0,0.300000
19,6,1,0.394261,1,0,0.300000


Pairwise ordering comparisons in this query: 10


#### Step 7: Offline ranking evaluation

**What:** Optional **console-only** metric: proxy **NDCG@K** (no human labels).

**How:** Compares ordering by **`trend_score`** to an “ideal” ordering by blended gain on the top-**K** rows (K = `llm_top_n`, capped by table length). Blended gain is `0.5*ctr_recency_norm + 0.3*volume_norm + 0.2*momentum_norm`, where `ctr_recency_norm` is based on `avg_proxy_ctr_recency`. Toggle with **`log_ranking_evaluation`** on `Settings`.

**Why:** Cheap **sanity check** that the ranker is aligned with engagement quality + scale + momentum together—not accuracy, not business KPIs. **How to read:** 1.0 = same order as sorting by blended gain on that slice; lower = more reordering.


In [12]:
_step_offline_ranking_evaluation(ctx)


Ranking evaluation (before LLM enrichment)
------------------------------------------------------------
Proxy NDCG (trend_score order vs ideal by blended gain)
  Interpretation: 1.0 = same ordering as sorting by blended gain; <1.0 = ranker reorders vs blended proxy.
  k: 10.0
  gain_formula: 0.5*ctr_recency_norm + 0.3*volume_norm + 0.2*momentum_norm
  score_col: trend_score
  dcg@k (score order): 3.73888855438753
  idcg@k (ideal by gain): 3.87578382655542
  ndcg@k (proxy): 0.9646793324153079
------------------------------------------------------------


#### Step 8: Attach topic keywords

**What:** Human-readable **labels and keyword lists** per topic from the **fitted** BERTopic model.

**How:** `add_topic_keyword_columns` fills `topic_keywords`, **`dominant_topic_keywords`**, and **`topic_label`** (short string from the first keywords).

**Why:** Step 9 needs lexical context for taxonomy/coherence checks, display names, and the LLM prompt; **dominant** keywords are what feed classification and the API text (see `enrich_topic_insights_marketer_fields`).


In [13]:
_step_attach_topic_keywords(ctx)

t = ctx.topic_insights_df
assert t is not None
cols = ["topic", "topic_label", "topic_keywords", "dominant_topic_keywords"]
cols = [c for c in cols if c in t.columns]
display(t[cols].head(5))

,topic,topic_label,topic_keywords,dominant_topic_keywords
0,0,"music, official, video, acoustic, camila","[music, official, video, acoustic, camila, harmony, ask, audio]","[music, official, acoustic, camila, harmony]"
1,8,"liza, google, koshy, shower, meme","[liza, google, koshy, shower, meme, mickey, bad, mouse]","[liza, google, koshy, shower, meme]"
2,0,"music, official, video, acoustic, camila","[music, official, video, acoustic, camila, harmony, ask, audio]","[music, official, acoustic, camila, harmony]"
3,8,"liza, google, koshy, shower, meme","[liza, google, koshy, shower, meme, mickey, bad, mouse]","[liza, google, koshy, shower, meme]"
4,11,"christmas, amazon, advert, lewis, anastasia","[christmas, amazon, advert, lewis, anastasia, disney, xmas, john]","[christmas, amazon, advert, lewis, anastasia]"


#### Step 9: Marketer insights

**What:** Enrich each topic with **taxonomy**, **coherence**, sample titles, optional **LLM** JSON (`summary`, `campaign_copy`, …) for the first **`llm_top_n`** rows (by dataframe index).

**How:** Rule-based checks (`topic_is_coherent`, `classify_trend_type`, `TopicNamer`); if coherent and safe path, **`InsightGenerator.generate_insight`** calls OpenAI with `TREND_INSIGHT_PROMPT_TEMPLATE`; otherwise canned copy.

**Why:** Turns clusters into **marketer-facing** text; the code cell’s **truncated prompt** preview matches `InsightGenerator` for the top **`trend_score`** row (API may still skip incoherent rows).


In [14]:
_step_marketer_insights(ctx)

from src.constants.llm_prompts import TREND_INSIGHT_PROMPT_TEMPLATE

t = ctx.topic_insights_df
assert t is not None
print("topic_insights_df shape:", t.shape)
cols = [
    "topic",
    "topic_label",
    "ranker",
    "trend_score",
    "volume",
    "momentum",
    "avg_views",
    "avg_likes",
    "avg_proxy_ctr_recency",
    "marketing_safe",
]
cols = [c for c in cols if c in t.columns]
display(t[cols].head(8))

row0 = t.iloc[0]
llm_input = TREND_INSIGHT_PROMPT_TEMPLATE.format(
    topic_keywords=row0["dominant_topic_keywords"],
    sample_titles=row0["sample_titles"],
    trend_type=row0["trend_type"],
    avg_views=int(row0["avg_views"]),
    avg_likes=int(row0["avg_likes"]),
    momentum=f"{row0['momentum']:.2f}",
    avg_proxy_ctr_recency=f"{float(row0.get('avg_proxy_ctr_recency', 0.0)):.4f}",
)
print()
print("Sample LLM input (top `trend_score` row; truncated)")
print("=" * 60)
_max = 2400
print(llm_input[:_max] + ("" if len(llm_input) <= _max else "\n…"))
if len(llm_input) > _max:
    print(f"(truncated; full string is {len(llm_input)} chars — see TREND_INSIGHT_PROMPT_TEMPLATE in src/constants/llm_prompts.py)")

topic_insights_df shape: (25, 36)


,topic,topic_label,trend_score,volume,momentum,avg_views,avg_likes,avg_proxy_ctr_recency,marketing_safe
0,0,"music, official, video, acoustic, camila",1.000000,133,-0.034483,2.295221e+06,108496.203008,0.026410,True
1,8,"liza, google, koshy, shower, meme",0.976306,42,0.272727,6.187073e+05,33974.190476,0.029558,True
2,0,"music, official, video, acoustic, camila",0.967521,133,-0.034483,2.295221e+06,108496.203008,0.026410,True
3,8,"liza, google, koshy, shower, meme",0.952986,42,0.272727,6.187073e+05,33974.190476,0.029558,True
4,11,"christmas, amazon, advert, lewis, anastasia",0.915100,29,0.142857,1.305098e+06,28731.620690,0.033452,True
5,8,"liza, google, koshy, shower, meme",0.896756,42,0.272727,6.187073e+05,33974.190476,0.029558,True
6,3,"makeup, beauty, tan, fish, fashion",0.871057,58,-0.142857,5.361668e+05,22950.965517,0.021788,True
7,2,"iphone, face, react, life, pixel",0.827368,60,0.000000,8.673474e+05,19594.666667,0.019960,True



Sample LLM input (top `trend_score` row; truncated)

You generate grounded marketing insights for a Mailchimp-style trend engine.

Return ONLY valid JSON with these keys:
- summary
- campaign_angle
- suggested_subject
- email_hook
- marketing_safe

Inputs:
- topic_keywords: ['music', 'official', 'acoustic', 'camila', 'harmony']
- sample_titles: ["(SPOILERS) 'Shiva Saves the Day' Talked About Scene Ep. 804 | The Walking Dead", 'Eminem - Walk On Water (Audio) ft. Beyoncé', 'Hunter Hayes - You Should Be Loved (Part One Of Pictures)']
- trend_type: entertainment
- avg_views: 2295220
- avg_likes: 108496
- momentum: -0.03
- avg_proxy_ctr_recency: 0.0264

Rules:
1. Use only the provided inputs.
2. If a claim cannot be directly supported by topic_keywords or sample_titles, do not include it.
3. Do not invent specific shows, products, events, controversies, relationships, or storylines.
4. Treat names as context, not the main theme unless clearly dominant across multiple keywords or multiple s

#### Step 10: Save final outputs

**What:** Persist the **final** video-level and topic-level tables for the app and reuse.

**How:** `save_final_artifacts` validates rows and writes **`videos_with_topics.csv`** and **`topic_insights.csv`** under `output_dir` (paths in `src/constants/pipeline_io.py`).

**Why:** **`streamlit run app.py`** and `python -m src.evaluation` expect these paths; reproducible handoff from batch run to UI.


In [15]:
_step_save_final_artifacts(ctx)

out = Path(settings.output_dir)
vwt = out / VIDEOS_WITH_TOPICS_FILENAME
tic = out / TOPIC_INSIGHTS_FILENAME
print("videos_with_topics:", vwt.resolve(), vwt.is_file())
print("topic_insights:", tic.resolve(), tic.is_file())

videos_with_topics_df = ctx.videos_df
topic_insights_df = ctx.topic_insights_df

Saved final outputs to: /Users/s0s0rn1/Desktop/Intuit/Mailchimp-Email-Marketing/outputs
videos_with_topics: /Users/s0s0rn1/Desktop/Intuit/Mailchimp-Email-Marketing/outputs/videos_with_topics.csv True
topic_insights: /Users/s0s0rn1/Desktop/Intuit/Mailchimp-Email-Marketing/outputs/topic_insights.csv True


### Final output — top trends (pretty-print)

**What:** Same **pretty-print** loop as `main.py`: walk **`topic_insights_df`** in **`trend_score`** order and print scores plus **`summary`** / **`campaign_copy`**.

**Why:** Quick **human-readable** demo of the end product without opening Streamlit. Does not re-run the pipeline.


In [16]:
assert videos_with_topics_df is not None and topic_insights_df is not None

top_n = settings.top_n_topics_to_show

for _, row in topic_insights_df.head(top_n).iterrows():
    suggestion = row["campaign_copy"]
    print("\n" + "=" * 60)
    print(f"Trend: {row['topic_label']}")
    print(f"Trend Score: {row['trend_score']:.2f}")
    print(f"Volume: {int(row['volume'])}")
    print(f"Avg Views: {int(row['avg_views']):,}")
    print(f"Avg Likes: {int(row['avg_likes']):,}")
    print(f"Momentum: {row['momentum']:.2f}")
    print("\nSummary:")
    print(row["summary"])
    print("\nCampaign copy:")
    print(f"  Suggested Subject: {suggestion['suggested_subject']}")
    print(f"  Campaign Angle: {suggestion['campaign_angle']}")
    print(f"  Email Hook: {suggestion['email_hook']}")


Trend: music, official, video, acoustic, camila
Trend Score: 1.00
Volume: 133
Avg Views: 2,295,220
Avg Likes: 108,496
Momentum: -0.03

Summary:
The entertainment trend cluster centers on acoustic music content with an official vibe, incorporating harmony and emotional appeal. Engagement is steady with significant average views and likes, indicating a consistent interest in authentic acoustic performances.

Campaign copy:
  Suggested Subject: Exclusive Acoustic Harmony Music Releases
  Campaign Angle: Leverage the popularity of acoustic and harmonious music by promoting official music releases and exclusive acoustic sessions to engage dedicated music fans.
  Email Hook: Discover the soulful blend of official acoustic tracks bringing harmony to your playlist now!

Trend: liza, google, koshy, shower, meme
Trend Score: 0.98
Volume: 42
Avg Views: 618,707
Avg Likes: 33,974
Momentum: 0.27

Summary:
Content blending everyday relatable humor with recognizable brand and cultural references cont